# 01 - Data Acquisition
Downloads Cricsheet ball-by-ball data and scrapes ESPNcricinfo Statsguru innings-level stats for India.

In [8]:
import sys
sys.path.append('../src')
from india_cricket_downloader_v4 import download_cricsheet, scrape_espncricinfo
from pathlib import Path

OUTPUT_DIR = Path('../data/raw')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1.1 Cricsheet Ball-by-Ball Data

In [9]:
download_cricsheet(OUTPUT_DIR)

  [skip] cricsheet_tests.csv already exists
  [skip] cricsheet_odis.csv already exists
  [skip] cricsheet_t20is.csv already exists
  [skip] cricsheet_ipl.csv already exists


## 1.2 ESPNcricinfo Statsguru
Scrapes innings-level batting and bowling tables for Test, ODI, and T20I formats.

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {'User-Agent': 'Mozilla/5.0'}

def scrape_statsguru(base_url, pages=10):
    """Scrape paginated Statsguru innings table."""
    all_rows = []
    for page in range(1, pages + 1):
        url = f"{base_url};page={page}"
        resp = requests.get(url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            break
        soup = BeautifulSoup(resp.text, 'lxml')
        table = soup.find('table', class_='engineTable')
        if table is None:
            break
        rows = table.find_all('tr', class_=['data1', 'data2'])
        if not rows:
            break
        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all('td')]
            all_rows.append(cols)
        time.sleep(1.0)
    return all_rows

print('Scraping logic ready. Run scrape_statsguru() with target URLs.')

Scraping logic ready. Run scrape_statsguru() with target URLs.


## 1.3 Verify Downloads

In [11]:
import os
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    size = os.path.getsize(f) / 1024
    print(f"{f.name:45s}  {size:8.1f} KB")

cricsheet_ipl.csv                               39323.9 KB
cricsheet_odis.csv                              33384.9 KB
cricsheet_t20is.csv                              7655.0 KB
cricsheet_tests.csv                             49913.4 KB
espn_india_odi_batting.csv                        678.6 KB
espn_india_odi_batting_career.csv                   3.4 KB
espn_india_odi_bowling.csv                        669.0 KB
espn_india_odi_bowling_career.csv                   3.5 KB
espn_india_t20_batting.csv                        205.7 KB
espn_india_t20_batting_career.csv                   3.5 KB
espn_india_t20_bowling.csv                        198.0 KB
espn_india_t20_bowling_career.csv                   3.5 KB
espn_india_test_batting.csv                       684.5 KB
espn_india_test_batting_career.csv                  2.9 KB
espn_india_test_bowling.csv                       682.1 KB
espn_india_test_bowling_career.csv                  3.9 KB
